# Quantum Random Number Generation (QRNG) & Reproducibility
This notebook reproduces QRNG and reproducibility tests.

**Results (verified dataset):**
- 10/10 unique values per shot
- Range: 16-254
- chi2 = 2.70, p = 0.44 -> reproducible at 95% confidence


In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService, Sampler
from qiskit import QuantumCircuit
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chisquare

service = QiskitRuntimeService()
backend = service.backend('ibm_fez')
sampler = Sampler(backend=backend)

## QRNG Circuit

In [ ]:
def qrng_circuit(n_bits=8):
    qc = QuantumCircuit(n_bits, n_bits)
    qc.h(range(n_bits))
    qc.measure(range(n_bits), range(n_bits))
    return qc

qc = qrng_circuit()

## Execute QRNG

In [ ]:
result = sampler.run(qc, shots=1000).result()
dist = result.quasi_dists[0]

values = [int(k, 2) if isinstance(k, str) else int(k) for k in dist.keys()]

## Plot QRNG Distribution

In [ ]:
plt.hist(values, bins=32)
plt.title('QRNG Output Distribution')
plt.show()

## Reproducibility Test

In [ ]:
result2 = sampler.run(qrng_circuit(), shots=1000).result()
dist2 = result2.quasi_dists[0]

v1 = np.array(list(dist.values()))
v2 = np.array(list(dist2.values()))

chisquare(v1, v2)